<a href="https://colab.research.google.com/github/brandonwolk/kaneohe-coral-mcda/blob/main/DegreeHeatingWeeks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# [CELL: CLEAR_AND_MOUNT]
import os
import shutil

mount_point = '/content/drive'

# Clear out any local files trapped in the mountpoint path
if os.path.exists(mount_point):
    try:
        shutil.rmtree(mount_point)
        print("Cleared local directory blocking the mountpoint.")
    except Exception as e:
        print(f"Could not remove directly: {e}")

# Now mount Google Drive properly
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Cleared local directory blocking the mountpoint.
Mounted at /content/drive


In [ ]:
# [CELL: VERIFY_DRIVE_FILES]
import os

target_dir = '/content/drive/MyDrive/Capstone/01 Raw Data'

if os.path.exists(target_dir):
    files = os.listdir(target_dir)
    print(f"SUCCESS: Found {len(files)} files in Google Drive path!")
    print("Sample files:", files[:5])
else:
    print("WARNING: Directory path does not exist. Google Drive is likely NOT mounted properly!")

SUCCESS: Found 36 files in Google Drive path!
Sample files: ['kaneohe_bathy_4m.zip', 'hi_noaa_oahu_benthic_habitats.zip', '2007_HI_KaneoheBay_H_ASC.txt.xml', 'kaneohe_MB_Lidar_4m.grd', 'kaneohe_MB_Lidar_4m.asc']


In [ ]:
import os
import time
import calendar
import requests
import xarray as xr

class ERDDAPChunkedDownloader:
    """
    Downloads NOAA Coral Reef Watch 5km Daily DHW/SST data from the PacIOOS
    ERDDAP griddap server in yearly chunks, with a monthly fallback for
    years that hit server-side cache errors (HTTP 500 / NoSuchFileException).
    """

    def __init__(self,
                 base_url="https://pae-paha.pacioos.hawaii.edu/erddap/griddap/dhw_5km.nc",
                 lat_bounds=(21.35, 21.65),
                 lon_bounds=(-157.95, -157.70),
                 variables=("CRW_DHW", "CRW_SST"),
                 start_year=2000,
                 end_year=2025,
                 output_dir="/content/drive/MyDrive/Capstone/01 Raw Data/",
                 max_retries=5,
                 initial_backoff=10,
                 request_timeout=120):
        self.base_url = base_url
        self.lat_min, self.lat_max = lat_bounds
        self.lon_min, self.lon_max = lon_bounds
        self.variables = variables
        self.start_year = start_year
        self.end_year = end_year
        self.output_dir = output_dir
        self.max_retries = max_retries
        self.initial_backoff = initial_backoff
        self.request_timeout = request_timeout
        self._ensure_output_dir()

    def _ensure_output_dir(self):
        os.makedirs(self.output_dir, exist_ok=True)

    def _build_query_url(self, year):
        start_date = f"{year}-01-01T00:00:00Z"
        end_date = f"{year}-12-31T23:59:59Z"
        var_queries = [
            f"{var}[({start_date}):1:({end_date})]"
            f"[({self.lat_min}):1:({self.lat_max})]"
            f"[({self.lon_min}):1:({self.lon_max})]"
            for var in self.variables
        ]
        return f"{self.base_url}?{','.join(var_queries)}"

    def _output_filepath(self, year):
        return os.path.join(self.output_dir, f"kaneohe_bay_CRW_{year}.nc")

    def _download_year(self, year):
        url = self._build_query_url(year)
        filepath = self._output_filepath(year)

        if os.path.exists(filepath):
            print(f"[SKIP] {year}: file already exists at {filepath}")
            return True

        backoff = self.initial_backoff
        for attempt in range(1, self.max_retries + 1):
            try:
                print(f"[FETCH] {year}: attempt {attempt}/{self.max_retries} ...")
                response = requests.get(url, timeout=self.request_timeout, stream=True)

                if response.status_code == 200:
                    with open(filepath, "wb") as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"[OK]    {year}: saved to {filepath}")
                    return True

                elif response.status_code in (408, 500):
                    print(f"[{response.status_code}]   {year}: retryable error, backing off {backoff}s ...")
                    time.sleep(backoff)
                    backoff *= 2

                else:
                    print(f"[ERROR] {year}: HTTP {response.status_code} - {response.text[:300]}")
                    return False

            except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
                print(f"[TIMEOUT/CONN] {year}: {e}, backing off {backoff}s ...")
                time.sleep(backoff)
                backoff *= 2

            except requests.exceptions.RequestException as e:
                print(f"[FAIL]  {year}: unrecoverable exception: {e}")
                return False

        print(f"[GIVE UP] {year}: exceeded max retries ({self.max_retries})")
        return False

    def download_all(self):
        results = {"success": [], "failed": []}
        for year in range(self.start_year, self.end_year + 1):
            success = self._download_year(year)
            results["success" if success else "failed"].append(year)
            time.sleep(2)
        self._print_summary(results)
        return results

    def download_years(self, years):
        """Download only a specific list of years, e.g. [2024, 2025]."""
        results = {"success": [], "failed": []}
        for year in years:
            success = self._download_year(year)
            results["success" if success else "failed"].append(year)
            time.sleep(2)
        self._print_summary(results)
        return results

    def _print_summary(self, results):
        print("\n" + "=" * 50)
        print("DOWNLOAD SUMMARY")
        print("=" * 50)
        print(f"Succeeded ({len(results['success'])}): {results['success']}")
        print(f"Failed    ({len(results['failed'])}): {results['failed']}")
        print("=" * 50)

    def _build_monthly_query_url(self, year, month):
        last_day = calendar.monthrange(year, month)[1]
        start_date = f"{year}-{month:02d}-01T00:00:00Z"
        end_date = f"{year}-{month:02d}-{last_day:02d}T23:59:59Z"
        var_queries = [
            f"{var}[({start_date}):1:({end_date})]"
            f"[({self.lat_min}):1:({self.lat_max})]"
            f"[({self.lon_min}):1:({self.lon_max})]"
            for var in self.variables
        ]
        return f"{self.base_url}?{','.join(var_queries)}"

    def _download_month(self, year, month):
        url = self._build_monthly_query_url(year, month)
        filename = f"kaneohe_bay_CRW_{year}_{month:02d}.nc"
        filepath = os.path.join(self.output_dir, filename)

        if os.path.exists(filepath):
            print(f"[SKIP] {year}-{month:02d}: file already exists")
            return True

        backoff = self.initial_backoff
        for attempt in range(1, self.max_retries + 1):
            try:
                print(f"[FETCH] {year}-{month:02d}: attempt {attempt}/{self.max_retries} ...")
                response = requests.get(url, timeout=self.request_timeout, stream=True)

                if response.status_code == 200:
                    with open(filepath, "wb") as f:
                        for chunk in response.iter_content(chunk_size=8192):
                            f.write(chunk)
                    print(f"[OK]    {year}-{month:02d}: saved to {filepath}")
                    return True

                elif response.status_code in (408, 500):
                    print(f"[{response.status_code}]   {year}-{month:02d}: retryable error, backing off {backoff}s ...")
                    time.sleep(backoff)
                    backoff *= 2

                else:
                    print(f"[ERROR] {year}-{month:02d}: HTTP {response.status_code} - {response.text[:300]}")
                    return False

            except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
                print(f"[TIMEOUT/CONN] {year}-{month:02d}: {e}, backing off {backoff}s ...")
                time.sleep(backoff)
                backoff *= 2

            except requests.exceptions.RequestException as e:
                print(f"[FAIL]  {year}-{month:02d}: unrecoverable exception: {e}")
                return False

        print(f"[GIVE UP] {year}-{month:02d}: exceeded max retries")
        return False

    def download_year_monthly_fallback(self, year, end_month=None):
        last_month = end_month if end_month else 12
        results = {"success": [], "failed": []}
        for month in range(1, last_month + 1):
            success = self._download_month(year, month)
            results["success" if success else "failed"].append(month)
            time.sleep(2)
        print(f"\n[YEAR {year} MONTHLY SUMMARY] Succeeded: {results['success']} | Failed: {results['failed']}")
        return results

    def merge_chunks(self, output_filename="kaneohe_bay_CRW_2000_2025_merged.nc"):
        filepaths = [
            self._output_filepath(year)
            for year in range(self.start_year, self.end_year + 1)
            if os.path.exists(self._output_filepath(year))
        ]
        if not filepaths:
            print("[MERGE] No downloaded files found to merge.")
            return None
        print(f"[MERGE] Opening {len(filepaths)} files ...")
        ds = xr.open_mfdataset(filepaths, combine="by_coords")
        merged_path = os.path.join(self.output_dir, output_filename)
        ds.to_netcdf(merged_path)
        print(f"[MERGE] Saved merged dataset to {merged_path}")
        return ds

In [ ]:
downloader = ERDDAPChunkedDownloader(
    start_year=2000,
    end_year=2025,
    max_retries=5,
    initial_backoff=10,
    request_timeout=120
)

In [ ]:
results = downloader.download_years([2024, 2025])
print(results)

[FETCH] 2024: attempt 1/5 ...
[OK]    2024: saved to /content/drive/MyDrive/Capstone/01 Raw Data/kaneohe_bay_CRW_2024.nc
[FETCH] 2025: attempt 1/5 ...
[OK]    2025: saved to /content/drive/MyDrive/Capstone/01 Raw Data/kaneohe_bay_CRW_2025.nc

DOWNLOAD SUMMARY
Succeeded (2): [2024, 2025]
Failed    (0): []
{'success': [2024, 2025], 'failed': []}


In [ ]:
!pip install rioxarray

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.4/72.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 30.3 MB/s eta 0:00:00
  Attempting uninstall: xarray
    Found existing installation: xarray 2025.12.0
    Uninstalling xarray-2025.12.0:
      Successfully uninstalled xarray-2025.12.0


In [ ]:
# [CELL: FIXED_MERGE_AND_EXPORT_DHW_GEOTIFF]
import xarray as xr
import os
import rioxarray

output_dir = "/content/drive/MyDrive/Capstone/01 Raw Data"

print("1. Loading and combining all historical DHW chunk files from Google Drive...")
ds = xr.open_mfdataset(os.path.join(output_dir, "dhw_5km_CRW_DHW_*.nc"), combine="by_coords")

print("2. Computing maximum historical thermal stress (2000-2025)...")
summary_dhw = ds['CRW_DHW'].max(dim='time')

# Automatically detect spatial dimension names (handles 'lon' vs 'longitude')
x_name = 'lon' if 'lon' in summary_dhw.dims else 'longitude'
y_name = 'lat' if 'lat' in summary_dhw.dims else 'latitude'
print(f"Detected spatial dimensions: x={x_name}, y={y_name}")

print("3. Setting spatial dimensions and coordinate system (EPSG:4326)...")
summary_dhw = summary_dhw.rio.set_spatial_dims(x_dim=x_name, y_dim=y_name, inplace=True)
if summary_dhw.rio.crs is None:
    summary_dhw = summary_dhw.rio.write_crs("EPSG:4326", inplace=True)

# Define output file path
output_tif = os.path.join(output_dir, "kaneohe_bay_CRW_DHW_2000_2025_max.tif")

print(f"4. Exporting summary raster to GeoTIFF...")
summary_dhw.rio.to_raster(output_tif)

# Clean up
ds.close()
print(f"✅ Success! Your GIS-ready GeoTIFF is saved at:\n{output_tif}")

1. Loading and combining all historical DHW chunk files from Google Drive...
2. Computing maximum historical thermal stress (2000-2025)...
Detected spatial dimensions: x=longitude, y=latitude
3. Setting spatial dimensions and coordinate system (EPSG:4326)...
4. Exporting summary raster to GeoTIFF...
✅ Success! Your GIS-ready GeoTIFF is saved at:
/content/drive/MyDrive/Capstone/01 Raw Data/kaneohe_bay_CRW_DHW_2000_2025_max.tif


In [ ]:
import os
import xarray as xr

RAW_DATA_DIR = "/content/drive/MyDrive/Capstone/01 Raw Data"

print("--- All files in Raw Data directory ---")
all_files = sorted(os.listdir(RAW_DATA_DIR))
for f in all_files:
    full_path = os.path.join(RAW_DATA_DIR, f)
    size_kb = os.path.getsize(full_path) / 1024
    print(f"{f}  ({size_kb:.1f} KB)")

--- All files in Raw Data directory ---
2007_HI_KaneoheBay_H_ASC.txt.xml  (9.2 KB)
CDIP_Station198_WaveData.csv  (234488.0 KB)
PacIOOS_CWB_WaterQuality.csv  (372.2 KB)
cwb_water_quality.csv  (384.4 KB)
dhw_5km_CRW_DHW_full_timeseries.nc  (0.2 KB)
dhw_5km_CRW_SST_2000.nc  (57.9 KB)
dhw_5km_CRW_SST_full_timeseries.nc  (0.0 KB)
hi_noaa_oahu_benthic_habitats.zip  (4774.8 KB)
kaneohe_MB_Lidar_4m.asc  (85601.5 KB)
kaneohe_MB_Lidar_4m.grd  (53145.0 KB)
kaneohe_bathy_4m.jpg  (600.4 KB)
kaneohe_bathy_4m.pdf  (774.8 KB)
kaneohe_bathy_4m.txt  (12.1 KB)
kaneohe_bathy_4m.zip  (28191.9 KB)
kaneohe_bay_CRW_2000.nc  (288.8 KB)
kaneohe_bay_CRW_2001.nc  (288.1 KB)
kaneohe_bay_CRW_2002.nc  (288.1 KB)
kaneohe_bay_CRW_2003.nc  (288.1 KB)
kaneohe_bay_CRW_2004.nc  (288.8 KB)
kaneohe_bay_CRW_2005.nc  (288.1 KB)
kaneohe_bay_CRW_2006.nc  (288.1 KB)
kaneohe_bay_CRW_2007.nc  (288.1 KB)
kaneohe_bay_CRW_2008.nc  (288.8 KB)
kaneohe_bay_CRW_2009.nc  (288.1 KB)
kaneohe_bay_CRW_2010.nc  (288.1 KB)
kaneohe_bay_CRW_2011.

In [ ]:
print("\n--- Loading local DHW .nc files ---")

dhw_files = sorted([
    os.path.join(RAW_DATA_DIR, f)
    for f in os.listdir(RAW_DATA_DIR)
    if f.startswith("kaneohe_bay_CRW_") and f.endswith(".nc")
    and "SST" not in f  # excludes the dedicated SST files, keeps the combined DHW+SST ones
])

print(f"Found {len(dhw_files)} DHW files.")
for f in dhw_files:
    print(f"  {os.path.basename(f)}")

dhw_ds = xr.open_mfdataset(dhw_files, combine="by_coords")["CRW_DHW"]
print(f"\nCombined time range: {dhw_ds.time.min().values} to {dhw_ds.time.max().values}")
print(f"Total time steps: {dhw_ds.sizes['time']}")


--- Loading local DHW .nc files ---
Found 26 DHW files.
  kaneohe_bay_CRW_2000.nc
  kaneohe_bay_CRW_2001.nc
  kaneohe_bay_CRW_2002.nc
  kaneohe_bay_CRW_2003.nc
  kaneohe_bay_CRW_2004.nc
  kaneohe_bay_CRW_2005.nc
  kaneohe_bay_CRW_2006.nc
  kaneohe_bay_CRW_2007.nc
  kaneohe_bay_CRW_2008.nc
  kaneohe_bay_CRW_2009.nc
  kaneohe_bay_CRW_2010.nc
  kaneohe_bay_CRW_2011.nc
  kaneohe_bay_CRW_2012.nc
  kaneohe_bay_CRW_2013.nc
  kaneohe_bay_CRW_2014.nc
  kaneohe_bay_CRW_2015.nc
  kaneohe_bay_CRW_2016.nc
  kaneohe_bay_CRW_2017.nc
  kaneohe_bay_CRW_2018.nc
  kaneohe_bay_CRW_2019.nc
  kaneohe_bay_CRW_2020.nc
  kaneohe_bay_CRW_2021.nc
  kaneohe_bay_CRW_2022.nc
  kaneohe_bay_CRW_2023.nc
  kaneohe_bay_CRW_2024.nc
  kaneohe_bay_CRW_2025.nc

Combined time range: 2000-01-01T12:00:00.000000000 to 2025-12-31T12:00:00.000000000
Total time steps: 9492


In [ ]:
import rioxarray

BATHY_PATH = "/content/drive/MyDrive/Capstone/02 Processed Rasters/Kaneohe_Bathy_Final.tif"
PROCESSED_DIR = "/content/drive/MyDrive/Capstone/02 Processed Rasters"

bathy_da = rioxarray.open_rasterio(BATHY_PATH, masked=True).squeeze()
print(f"Bathy CRS: {bathy_da.rio.crs}")
print(f"Bathy shape: {bathy_da.rio.shape}")

Bathy CRS: EPSG:6634
Bathy shape: (3853, 3531)


In [ ]:
print("--- Computing max DHW across time ---")
dhw_max = dhw_ds.max(dim="time", skipna=True)
print(f"DHW max computed — shape: {dhw_max.shape}")

--- Computing max DHW across time ---
DHW max computed — shape: (8, 6)


In [ ]:
print("--- Reprojecting and clipping to bathymetry grid ---")

dhw_max_geo = dhw_max.rename({"longitude": "x", "latitude": "y"})
dhw_max_geo = dhw_max_geo.rio.write_crs("EPSG:4326", inplace=False)
dhw_max_geo = dhw_max_geo.rio.set_spatial_dims(x_dim="x", y_dim="y", inplace=False)

dhw_max_aligned = dhw_max_geo.rio.reproject_match(bathy_da)

print(f"Aligned shape: {dhw_max_aligned.rio.shape}")
print(f"Aligned CRS: {dhw_max_aligned.rio.crs}")
print(f"Aligned bounds: {dhw_max_aligned.rio.bounds()}")

--- Reprojecting and clipping to bathymetry grid ---
Aligned shape: (3853, 3531)
Aligned CRS: EPSG:6634
Aligned bounds: (618856.77, 2367957.47, 632980.77, 2383369.47)


In [ ]:
output_path = f"{PROCESSED_DIR}/kaneohe_bay_CRW_DHW_2000_2025_max.tif"
dhw_max_aligned.rio.to_raster(output_path)
print(f"[SAVED] {output_path}")

[SAVED] /content/drive/MyDrive/Capstone/02 Processed Rasters/kaneohe_bay_CRW_DHW_2000_2025_max.tif


In [11]:
from google.colab import drive
import os
import xarray as xr
import rioxarray

# Mount drive safely
drive.mount('/content/drive', force_remount=True)

raw_dir = "/content/drive/MyDrive/Capstone/01 Raw Data"
processed_dir = "/content/drive/MyDrive/Capstone/02 Processed Rasters"
os.makedirs(processed_dir, exist_ok=True)

# Clean up the old broken 0MB file if it's still hanging around
broken_file = os.path.join(raw_dir, "dhw_5km_CRW_DHW_full_timeseries.nc")
if os.path.exists(broken_file) and os.path.getsize(broken_file) == 0:
    os.remove(broken_file)

print("Processing DHW...")
# Grab all yearly DHW files (excluding SST)
dhw_files = sorted([
    os.path.join(raw_dir, f) for f in os.listdir(raw_dir)
    if f.startswith('kaneohe_bay_CRW_') and 'SST' not in f and f.endswith('.nc')
])

# Open and combine all yearly local files automatically using netcdf4 engine
ds_dhw = xr.open_mfdataset(dhw_files, combine='by_coords', engine='netcdf4')

dhw_var = ds_dhw['CRW_DHW']
dhw_var = dhw_var.rio.set_spatial_dims(x_dim='longitude', y_dim='latitude', inplace=True)
dhw_var = dhw_var.rio.write_crs("EPSG:4326", inplace=True)

# Collapse time into one 25-year max file
summary_dhw = dhw_var.max(dim='time')
summary_dhw.rio.to_raster(os.path.join(processed_dir, "kaneohe_bay_CRW_DHW_2000_2025_max.tif"), overwrite=True)
ds_dhw.close()

print("✅ DHW Done! GeoTIFF saved to your Processed Rasters folder.")

Mounted at /content/drive
Processing DHW...


✅ DHW Done! GeoTIFF saved to your Processed Rasters folder.
